# Step 5 — Markdown Chunking

This notebook will:

1. Read Markdown content from the `pages` table.
2. Split documents based on Markdown headings (`#`, `##`, etc.).
3. Calculate token counts (based on word count).
4. Save results to the `chunks` table.

> At this stage we **do not generate embeddings yet**. The focus is only on producing chunks ready for embedding.

In [1]:
from pathlib import Path
import sqlite3
import re

DB_PATH = Path("../data/linux_docs.db")
assert DB_PATH.exists(), "Database tidak ditemukan."

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()


## Fetch pages with Markdown content

In [2]:
cursor.execute("""
SELECT id, title, markdown
FROM pages
WHERE markdown IS NOT NULL
  AND TRIM(markdown) <> '';
""")

pages = cursor.fetchall()
print(f"Total pages: {len(pages)}")

Total pages: 1


## Chunking function based on headings

In [3]:
HEADING_PATTERN = re.compile(r"^(#{1,6})\s+(.*)$")

def split_markdown(markdown: str):
    chunks = []
    current_section = "Introduction"
    buffer = []

    for line in markdown.splitlines():
        match = HEADING_PATTERN.match(line)

        if match:
            if buffer:
                chunks.append({
                    "section": current_section,
                    "content": "\n".join(buffer).strip()
                })
                buffer = []

            current_section = match.group(2).strip()
        else:
            buffer.append(line)

    if buffer:
        chunks.append({
            "section": current_section,
            "content": "\n".join(buffer).strip()
        })

    return [c for c in chunks if c["content"]]


## Save chunks to SQLite

In [4]:
cursor.execute("DELETE FROM chunks")

total_chunks = 0

for page in pages:

    chunks = split_markdown(page["markdown"])

    for idx, chunk in enumerate(chunks):

        token_count = len(chunk["content"].split())

        cursor.execute(
            '''
            INSERT INTO chunks(
                page_id,
                section,
                chunk_index,
                content,
                token_count
            )
            VALUES (?, ?, ?, ?, ?)
            ''',
            (
                page["id"],
                chunk["section"],
                idx,
                chunk["content"],
                token_count
            )
        )

        total_chunks += 1

conn.commit()

print(f"✅ Total chunks saved: {total_chunks}")

✅ Total chunks saved: 41


## Verification

In [5]:
cursor.execute("""
SELECT
    page_id,
    section,
    chunk_index,
    token_count,
    substr(content,1,200) AS preview
FROM chunks
ORDER BY page_id, chunk_index
LIMIT 10;
""")

for row in cursor.fetchall():
    print("="*80)
    print(dict(row))


{'page_id': 1, 'section': 'Introduction', 'chunk_index': 0, 'token_count': 185, 'preview': 'This document is a guide for installing [Arch Linux](/title/Arch_Linux) using the live system booted from an installation medium made from an official installation image. Its accessibility features ar'}
{'page_id': 1, 'section': 'Acquire an installation image', 'chunk_index': 1, 'token_count': 26, 'preview': 'Visit [https://archlinux.org/download](https://archlinux.org/download) for official images, and, depending on how you want to boot, acquire the ISO file or a netboot image, and the respective PGP sign'}
{'page_id': 1, 'section': 'Verify signature', 'chunk_index': 2, 'token_count': 153, 'preview': 'It is recommended to verify the image signature before use, especially when downloading from an *HTTP mirror*, where downloads are generally prone to be intercepted to [serve malicious images](https:/'}
{'page_id': 1, 'section': 'Prepare an installation medium', 'chunk_index': 3, 'token_count': 56

## Notes

This notebook uses a simple heading-based approach for Markdown chunking.

In Step 6 we will:
- Create embeddings for each chunk,
- Store embeddings,
- Prepare semantic search.

In [6]:
conn.close()
print("Done ✅")

Done ✅
